# 🚀 Huấn luyện MobileNetV3 (Waste Classification) trên Kaggle
Notebook này chứa toàn bộ quy trình: Khởi tạo dữ liệu -> Build Model (MobileNetV3) -> Huấn luyện 2 Phase -> Trích xuất TFLite (Float32 & INT8).

**Lưu ý khi chạy trên Kaggle:**
1. Bật **GPU T100 / P100** trong cài đặt Notebook (Accelerator: GPU P100).
2. Kiểm tra lại biến `DATA_DIR` ở Cell tiếp theo để trỏ đúng vào thư mục dataset bạn đã Upload.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# --- 1. CẤU HÌNH (CONFIG) ---

# SỬA ĐƯỜNG DẪN NÀY THÀNH ĐƯỜNG DẪN DATASET TRÊN KAGGLE CỦA BẠN
# Ví dụ: DATA_DIR = '/kaggle/input/smart-waste-dataset/train'
DATA_DIR = '/kaggle/input/smart-waste-dataset/train' # Thay đổi nếu cần!

# Thư mục lưu Output trên Kaggle (/kaggle/working/ là nơi cho phép ghi file)
OUTPUT_DIR = '/kaggle/working/'
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, 'mobilenetv3_waste_best.keras')
TFLITE_SAVE_PATH = os.path.join(OUTPUT_DIR, 'mobilenetv3_waste.tflite')

# Hyperparameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15           # Phase 1
FINE_TUNE_EPOCHS = 20 # Phase 2
NUM_CLASSES = 5

CLASS_NAMES = ['bag_other', 'metal', 'mixed', 'paper', 'plastic']

TEST_SPLIT = 0.1
VAL_SPLIT = 0.15
RANDOM_SEED = 16

In [ ]:
# --- 2. DATA LOADER ---

def get_data_augmentation():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomBrightness(0.1),
        tf.keras.layers.RandomContrast(0.1),
    ])

def load_all_filepaths_and_labels():
    filepaths = []
    labels = []
    
    if not os.path.exists(DATA_DIR):
        print(f"[LỖI] Không tìm thấy thư mục: {DATA_DIR}")
        print("HÃY CHẮC CHẮN BẠN ĐÃ ĐỔI `DATA_DIR` THÀNH ĐƯỜNG DẪN KAGGLE CỦA BẠN!")
        return np.array([]), np.array([])
        
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(DATA_DIR, class_name)
        if not os.path.exists(class_dir):
            continue
            
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                filepaths.append(os.path.join(class_dir, img_name))
                labels.append(class_idx)
                
    return np.array(filepaths), np.array(labels)

def get_test_set(filepaths, labels):
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        filepaths, labels, 
        test_size=TEST_SPLIT, 
        stratify=labels,
        random_state=RANDOM_SEED
    )
    return (X_train_val, y_train_val), (X_test, y_test)

def create_dataset_from_paths(filepaths, labels, is_training=False):
    def parse_function(filename, label):
        image_string = tf.io.read_file(filename)
        image = tf.image.decode_jpeg(image_string, channels=3)
        image = tf.image.resize(image, IMG_SIZE)
        return image, label

    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    dataset = dataset.map(parse_function, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.cache()
    
    if is_training:
        dataset = dataset.shuffle(buffer_size=1000, seed=RANDOM_SEED)
        
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

def get_train_val_splits(X_train_val, y_train_val):
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, 
        test_size=VAL_SPLIT, 
        stratify=y_train_val,
        random_state=RANDOM_SEED
    )
    return (X_train, y_train), (X_val, y_val)

In [ ]:
# --- 3. MODEL ARCHITECTURE ---

def build_mobilenetv3_model(learning_rate=0.001):
    base_model = tf.keras.applications.MobileNetV3Large(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False
    
    data_augmentation = get_data_augmentation()
    
    model = tf.keras.Sequential([
        data_augmentation,
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    return model, base_model

def unfreeze_and_recompile(model, base_model, learning_rate=1e-5):
    base_model.trainable = True
    fine_tune_at = len(base_model.layers) - 30
    
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False
        
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    return model

In [ ]:
# --- 4. MAIN TRAINING & EVALUATION ---

def plot_history(history_list):
    acc = history_list[0].history['accuracy'] + history_list[1].history['accuracy']
    val_acc = history_list[0].history['val_accuracy'] + history_list[1].history['val_accuracy']
    loss = history_list[0].history['loss'] + history_list[1].history['loss']
    val_loss = history_list[0].history['val_loss'] + history_list[1].history['val_loss']

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Training Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.ylabel('Cross Entropy')
    plt.title('Training and Validation Loss')
    plt.xlabel('epoch')
    
    os.makedirs(os.path.join(OUTPUT_DIR, 'image'), exist_ok=True)
    plt.savefig(os.path.join(OUTPUT_DIR, 'image/training_history.png'))
    plt.show()

print("1. Đang nạp danh sách dữ liệu...")
filepaths, labels = load_all_filepaths_and_labels()

if len(filepaths) > 0:
    print("2. Tách Test Set...")
    (X_train_val, y_train_val), (X_test, y_test) = get_test_set(filepaths, labels)
    
    print("3. Tách Train Set và Validation Set...")
    (X_train, y_train), (X_val, y_val) = get_train_val_splits(X_train_val, y_train_val)
    
    print(f"Tổng số ảnh: {len(filepaths)}")
    print(f"Số ảnh để Train: {len(X_train)}")
    print(f"Số ảnh để Validation: {len(X_val)}")
    print(f"Số ảnh cất đi làm Test Set: {len(X_test)}")
    
    np.save(os.path.join(OUTPUT_DIR, 'test_filepaths.npy'), X_test)
    np.save(os.path.join(OUTPUT_DIR, 'test_labels.npy'), y_test)

    print("\n4. Chuẩn bị dữ liệu Dataset...")
    train_ds = create_dataset_from_paths(X_train, y_train, is_training=True)
    val_ds = create_dataset_from_paths(X_val, y_val, is_training=False)
    
    model, base_model = build_mobilenetv3_model()
    
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(filepath=MODEL_SAVE_PATH, save_best_only=True, monitor='val_accuracy'),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
    ]
    
    print("\n--- Phase 1: Feature Extraction ---")
    history_p1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
    
    print("\n--- Phase 2: Fine-Tuning ---")
    model = unfreeze_and_recompile(model, base_model)
    history_p2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, callbacks=callbacks)
    
    print(f"\nĐánh giá cuối cùng trên Validation:")
    loss, val_acc = model.evaluate(val_ds)
    print(f"-> Độ chính xác: {val_acc*100:.2f}%")
    
    plot_history([history_p1, history_p2])
    
    print("\n--- Đánh giá mô hình trên Test Set ---")
    test_ds = create_dataset_from_paths(X_test, y_test, is_training=False)
    test_loss, test_acc = model.evaluate(test_ds)
    print(f"-> Độ chính xác trên Test Set: {test_acc*100:.2f}%\n")
    
    from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
    y_pred_probs = model.predict(test_ds)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    print("\nBáo cáo phân loại (Classification Report):")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
    plt.title('Confusion Matrix (Ma trận nhầm lẫn)')
    plt.savefig(os.path.join(OUTPUT_DIR, 'image/confusion_matrix.png'))
    plt.show()
    
    print("\n--- Rút ngẫu nhiên 6 tấm trong tập Test ra dự đoán thử ---")
    random_idx = np.random.choice(len(X_test), 6, replace=False)
    X_random = X_test[random_idx]
    y_random = y_test[random_idx]
    test_ds_rand = create_dataset_from_paths(X_random, y_random, is_training=False)
    
    for images, labels in test_ds_rand.take(1):
        predictions = model.predict(images, verbose=0)
        plt.figure(figsize=(12, 8))
        for i in range(6):
            if i >= len(images): break
            ax = plt.subplot(2, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            pred_idx = np.argmax(predictions[i])
            pred_prob = np.max(predictions[i]) * 100
            pred_class = CLASS_NAMES[pred_idx]
            actual_class = CLASS_NAMES[labels[i].numpy()]
            color = "green" if pred_class == actual_class else "red"
            plt.title(f"Dự đoán: {pred_class} ({pred_prob:.1f}%)\nThực tế: {actual_class}", color=color)
            plt.axis("off")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'image/test_predictions.png'))
        plt.show()
else:
    print("Hủy quá trình train do không tìm thấy dữ liệu.")

In [ ]:
# --- 5. EXPORT TFLITE & INT8 QUANTIZATION ---
if len(filepaths) > 0:
    print("5. Xuất mô hình sang TFLite (Float32)...")
    best_model = tf.keras.models.load_model(MODEL_SAVE_PATH)
    converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
    tflite_model = converter.convert()
    with open(TFLITE_SAVE_PATH, 'wb') as f:
        f.write(tflite_model)
        
    print("6. Thực hiện Lượng tử hóa (INT8)...")
    converter_int8 = tf.lite.TFLiteConverter.from_keras_model(best_model)
    converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
    
    def representative_dataset():
        for images, _ in val_ds.take(100):
            yield [images]
            
    converter_int8.representative_dataset = representative_dataset
    converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter_int8.inference_input_type = tf.uint8
    converter_int8.inference_output_type = tf.uint8

    tflite_quant_model = converter_int8.convert()
    int8_save_path = TFLITE_SAVE_PATH.replace('.tflite', '_INT8.tflite')
    with open(int8_save_path, 'wb') as f:
        f.write(tflite_quant_model)
            
    print("\n=== HOÀN THÀNH TẤT CẢ ===")
    print(f"Các file đã được lưu tại thư mục: {OUTPUT_DIR}")
    print(f"1. Mô hình Keras: {MODEL_SAVE_PATH}")
    print(f"2. TFLite Float32: {TFLITE_SAVE_PATH}")
    print(f"3. TFLite INT8: {int8_save_path}")
    print("Bạn có thể tải các file này xuống từ thư mục /working của Kaggle.")